In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"

model_path = f"{processed_path}/hist_gradient_boosting_model.joblib"

print("Model exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("Model size:", round(os.path.getsize(model_path) / 1024, 2), "KB")

Model exists: True
Model size: 1068.62 KB


In [ ]:
import os

project_path = "/content/drive/MyDrive/Consumer Complain Project"

app_path = os.path.join(project_path, "app")

os.makedirs(app_path, exist_ok=True)

print("App folder:")
print(app_path)

print("\nFolder exists:", os.path.exists(app_path))

App folder:
/content/drive/MyDrive/Consumer Complain Project/app

Folder exists: True


In [ ]:
app_file = os.path.join(app_path, "app.py")

app_code = '''import streamlit as st

st.set_page_config(
    page_title="Consumer Complaint Response Predictor",
    page_icon="📊",
    layout="centered"
)

st.title("Consumer Complaint Response Predictor")

st.write(
    "This application predicts the expected company response "
    "to a consumer complaint using a trained machine learning model."
)

st.info("Application setup is working.")
'''

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created:", app_file)
print("File exists:", os.path.exists(app_file))

Created: /content/drive/MyDrive/Consumer Complain Project/app/app.py
File exists: True


In [ ]:
!pip install -q streamlit

import streamlit

print("Streamlit version:", streamlit.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 96.6 MB/s eta 0:00:00
Streamlit version: 1.64.0


In [ ]:
app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

app_code = '''import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Consumer Complaint Response Predictor",
    page_icon="📊",
    layout="centered"
)

# Project paths
processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"

# Load trained model
model = joblib.load(
    f"{processed_path}/hist_gradient_boosting_model.joblib"
)

# Load preprocessing artifacts
encoder = joblib.load(
    f"{processed_path}/one_hot_encoder.joblib"
)

company_frequency_df = pd.read_csv(
    f"{processed_path}/company_frequency_mapping.csv"
)

company_frequency_mapping = dict(
    zip(
        company_frequency_df["company"],
        company_frequency_df["company_frequency"]
    )
)

# Load feature schema
feature_schema = pd.read_csv(
    f"{processed_path}/prediction_feature_schema.csv"
)

st.title("Consumer Complaint Response Predictor")

st.write(
    "This application predicts the expected company response "
    "to a consumer complaint using a trained machine learning model."
)

st.success("Model and preprocessing artifacts loaded successfully.")

st.write("Expected model features:", len(feature_schema))
'''

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Updated:", app_file)
print("File exists:", os.path.exists(app_file))

Updated: /content/drive/MyDrive/Consumer Complain Project/app/app.py
File exists: True


In [ ]:
import subprocess
import time

app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

process = subprocess.Popen(
    ["streamlit", "run", app_file, "--server.port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

print("Streamlit process started.")
print("Process ID:", process.pid)

Streamlit process started.
Process ID: 3045


In [ ]:
!pip install -q pyngrok

In [ ]:
import requests

response = requests.get("http://localhost:8501")

print("Status code:", response.status_code)
print("Streamlit app is running:", response.status_code == 200)

Status code: 200
Streamlit app is running: True


In [ ]:
app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

app_code = '''import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Consumer Complaint Response Predictor",
    page_icon="📊",
    layout="centered"
)

# --------------------------------------------------
# Paths
# --------------------------------------------------

processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"


# --------------------------------------------------
# Load trained artifacts
# --------------------------------------------------

model = joblib.load(
    f"{processed_path}/hist_gradient_boosting_model.joblib"
)

encoder = joblib.load(
    f"{processed_path}/one_hot_encoder.joblib"
)

company_frequency_df = pd.read_csv(
    f"{processed_path}/company_frequency_mapping.csv"
)

company_frequency_mapping = dict(
    zip(
        company_frequency_df["company"],
        company_frequency_df["company_frequency"]
    )
)

feature_schema = pd.read_csv(
    f"{processed_path}/prediction_feature_schema.csv"
)


# --------------------------------------------------
# Prediction function
# --------------------------------------------------

def predict_complaint_response(input_data):

    input_df = pd.DataFrame([input_data]).copy()

    # Date features
    input_df["date_received"] = pd.to_datetime(
        input_df["date_received"],
        errors="coerce"
    )

    input_df["received_year"] = input_df["date_received"].dt.year
    input_df["received_month"] = input_df["date_received"].dt.month
    input_df["received_dayofweek"] = input_df["date_received"].dt.dayofweek
    input_df["received_day"] = input_df["date_received"].dt.day
    input_df["received_quarter"] = input_df["date_received"].dt.quarter
    input_df["received_hour"] = input_df["date_received"].dt.hour

    # Narrative features
    narrative = (
        input_df["consumer_complaint_narrative"]
        .fillna("")
        .astype(str)
    )

    input_df["narrative_present"] = (
        narrative.str.strip().ne("").astype(int)
    )

    input_df["narrative_length"] = narrative.str.len()

    input_df["narrative_word_count"] = (
        narrative.str.split().str.len()
    )

    # Company frequency
    input_df["company_frequency"] = (
        input_df["company"]
        .map(company_frequency_mapping)
        .fillna(0)
    )

    # Numeric features
    numeric_features = [
        "received_year",
        "received_month",
        "received_dayofweek",
        "received_day",
        "received_quarter",
        "received_hour",
        "narrative_present",
        "narrative_length",
        "narrative_word_count",
        "company_frequency"
    ]

    input_numeric = input_df[numeric_features].copy()

    # Categorical features
    encoded_categorical_features = [
        "product",
        "sub_product",
        "issue",
        "sub_issue",
        "submitted_via",
        "state"
    ]

    input_categorical = input_df[
        encoded_categorical_features
    ].copy()

    # One-hot encoding
    input_encoded = encoder.transform(
        input_categorical
    )

    input_encoded_df = pd.DataFrame(
        input_encoded,
        columns=encoder.get_feature_names_out(
            encoded_categorical_features
        ),
        index=input_df.index
    )

    # Combine numeric + encoded features
    model_input = pd.concat(
        [input_numeric, input_encoded_df],
        axis=1
    )

    # Enforce exact training feature order
    model_input = model_input.reindex(
        columns=feature_schema["feature"],
        fill_value=0
    )

    # Prediction
    prediction = model.predict(model_input)[0]

    probabilities = model.predict_proba(model_input)[0]

    probability_table = pd.DataFrame({
        "response_class": model.classes_,
        "probability": probabilities
    }).sort_values(
        "probability",
        ascending=False
    ).reset_index(drop=True)

    return prediction, probability_table


# --------------------------------------------------
# Application
# --------------------------------------------------

st.title("Consumer Complaint Response Predictor")

st.write(
    "This application predicts the expected company response "
    "to a consumer complaint using a trained machine learning model."
)

st.success("Model and preprocessing artifacts loaded successfully.")

st.write(
    "Expected model features:",
    len(feature_schema)
)
'''

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("Updated app.py successfully.")
print("Prediction function added.")

Updated app.py successfully.
Prediction function added.


In [ ]:
app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

with open(app_file, "r", encoding="utf-8") as f:
    app_code = f.read()

ui_code = '''

# --------------------------------------------------
# User Input Form
# --------------------------------------------------

st.subheader("Complaint Details")

product = st.text_input(
    "Product",
    placeholder="Example: Credit card"
)

sub_product = st.text_input(
    "Sub-product",
    placeholder="Example: Credit card"
)

issue = st.text_input(
    "Issue",
    placeholder="Example: Billing disputes"
)

sub_issue = st.text_input(
    "Sub-issue",
    placeholder="Example: Billing dispute"
)

company = st.text_input(
    "Company",
    placeholder="Example: Example Company"
)

state = st.text_input(
    "State",
    placeholder="Example: CA"
)

submitted_via = st.selectbox(
    "Submitted via",
    ["Web", "Phone", "Referral", "Postal mail"]
)

date_received = st.date_input(
    "Date received"
)

consumer_complaint_narrative = st.text_area(
    "Consumer complaint narrative",
    placeholder="Describe the consumer complaint...",
    height=150
)
'''

# Add UI before the end of the application file
app_code += ui_code

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("UI input form added successfully.")

UI input form added successfully.


In [ ]:
app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

with open(app_file, "r", encoding="utf-8") as f:
    app_code = f.read()

print(app_code)

import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Consumer Complaint Response Predictor",
    page_icon="📊",
    layout="centered"
)

# --------------------------------------------------
# Paths
# --------------------------------------------------

processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"


# --------------------------------------------------
# Load trained artifacts
# --------------------------------------------------

model = joblib.load(
    f"{processed_path}/hist_gradient_boosting_model.joblib"
)

encoder = joblib.load(
    f"{processed_path}/one_hot_encoder.joblib"
)

company_frequency_df = pd.read_csv(
    f"{processed_path}/company_frequency_mapping.csv"
)

company_frequency_mapping = dict(
    zip(
        company_frequency_df["company"],
        company_frequency_df["company_frequency"]
    )
)

feature_schema = pd.read_csv(
    f"{processed_path}/prediction_feature_schema.csv"
)


# -------

In [ ]:
app_file = "/content/drive/MyDrive/Consumer Complain Project/app/app.py"

app_code = '''import streamlit as st
import pandas as pd
import joblib


# --------------------------------------------------
# Page Configuration
# --------------------------------------------------

st.set_page_config(
    page_title="Consumer Complaint Response Predictor",
    page_icon="📊",
    layout="centered"
)


# --------------------------------------------------
# Paths
# --------------------------------------------------

processed_path = "/content/drive/MyDrive/Consumer Complain Project/data/processed"


# --------------------------------------------------
# Load Trained Artifacts
# --------------------------------------------------

model = joblib.load(
    f"{processed_path}/hist_gradient_boosting_model.joblib"
)

encoder = joblib.load(
    f"{processed_path}/one_hot_encoder.joblib"
)

company_frequency_df = pd.read_csv(
    f"{processed_path}/company_frequency_mapping.csv"
)

company_frequency_mapping = dict(
    zip(
        company_frequency_df["company"],
        company_frequency_df["company_frequency"]
    )
)

feature_schema = pd.read_csv(
    f"{processed_path}/prediction_feature_schema.csv"
)


# --------------------------------------------------
# Prediction Function
# --------------------------------------------------

def predict_complaint_response(input_data):

    input_df = pd.DataFrame([input_data]).copy()

    # Date features
    input_df["date_received"] = pd.to_datetime(
        input_df["date_received"],
        errors="coerce"
    )

    input_df["received_year"] = input_df["date_received"].dt.year
    input_df["received_month"] = input_df["date_received"].dt.month
    input_df["received_dayofweek"] = input_df["date_received"].dt.dayofweek
    input_df["received_day"] = input_df["date_received"].dt.day
    input_df["received_quarter"] = input_df["date_received"].dt.quarter
    input_df["received_hour"] = input_df["date_received"].dt.hour

    # Narrative features
    narrative = (
        input_df["consumer_complaint_narrative"]
        .fillna("")
        .astype(str)
    )

    input_df["narrative_present"] = (
        narrative.str.strip().ne("").astype(int)
    )

    input_df["narrative_length"] = narrative.str.len()

    input_df["narrative_word_count"] = (
        narrative.str.split().str.len()
    )

    # Company frequency
    input_df["company_frequency"] = (
        input_df["company"]
        .map(company_frequency_mapping)
        .fillna(0)
    )

    # Numeric features
    numeric_features = [
        "received_year",
        "received_month",
        "received_dayofweek",
        "received_day",
        "received_quarter",
        "received_hour",
        "narrative_present",
        "narrative_length",
        "narrative_word_count",
        "company_frequency"
    ]

    input_numeric = input_df[numeric_features].copy()

    # Categorical features
    encoded_categorical_features = [
        "product",
        "sub_product",
        "issue",
        "sub_issue",
        "submitted_via",
        "state"
    ]

    input_categorical = input_df[
        encoded_categorical_features
    ].copy()

    # One-hot encoding
    input_encoded = encoder.transform(
        input_categorical
    )

    input_encoded_df = pd.DataFrame(
        input_encoded,
        columns=encoder.get_feature_names_out(
            encoded_categorical_features
        ),
        index=input_df.index
    )

    # Combine features
    model_input = pd.concat(
        [input_numeric, input_encoded_df],
        axis=1
    )

    # Match training feature order
    model_input = model_input.reindex(
        columns=feature_schema["feature"],
        fill_value=0
    )

    # Prediction
    prediction = model.predict(model_input)[0]

    probabilities = model.predict_proba(model_input)[0]

    probability_table = pd.DataFrame({
        "Response Class": model.classes_,
        "Probability": probabilities
    }).sort_values(
        "Probability",
        ascending=False
    ).reset_index(drop=True)

    probability_table["Probability"] = (
        probability_table["Probability"] * 100
    ).round(2)

    return prediction, probability_table


# --------------------------------------------------
# Application Header
# --------------------------------------------------

st.title("Consumer Complaint Response Predictor")

st.write(
    "Enter complaint details below to generate a predicted "
    "company response category using the trained machine learning model."
)


# --------------------------------------------------
# Complaint Input Form
# --------------------------------------------------

st.subheader("Complaint Details")

product = st.text_input(
    "Product",
    placeholder="Example: Credit card"
)

sub_product = st.text_input(
    "Sub-product",
    placeholder="Example: Credit card"
)

issue = st.text_input(
    "Issue",
    placeholder="Example: Billing disputes"
)

sub_issue = st.text_input(
    "Sub-issue",
    placeholder="Example: Billing dispute"
)

company = st.text_input(
    "Company",
    placeholder="Example: Example Company"
)

state = st.text_input(
    "State",
    placeholder="Example: CA"
)

submitted_via = st.selectbox(
    "Submitted via",
    ["Web", "Phone", "Referral", "Postal mail"]
)

date_received = st.date_input(
    "Date received"
)

consumer_complaint_narrative = st.text_area(
    "Consumer complaint narrative",
    placeholder="Describe the consumer complaint...",
    height=150
)


# --------------------------------------------------
# Prediction
# --------------------------------------------------

if st.button(
    "Predict Company Response",
    type="primary",
    use_container_width=True
):

    required_fields = {
        "Product": product,
        "Sub-product": sub_product,
        "Issue": issue,
        "Sub-issue": sub_issue,
        "Company": company,
        "State": state
    }

    missing_fields = [
        field
        for field, value in required_fields.items()
        if not value.strip()
    ]

    if missing_fields:

        st.warning(
            "Please complete the following fields: "
            + ", ".join(missing_fields)
        )

    else:

        input_data = {
            "product": product,
            "sub_product": sub_product,
            "issue": issue,
            "sub_issue": sub_issue,
            "company": company,
            "state": state,
            "submitted_via": submitted_via,
            "date_received": date_received,
            "consumer_complaint_narrative": consumer_complaint_narrative
        }

        prediction, probability_table = (
            predict_complaint_response(input_data)
        )

        st.subheader("Prediction")

        st.success(
            f"Predicted company response: **{prediction}**"
        )

        st.subheader("Prediction Probabilities")

        st.dataframe(
            probability_table,
            use_container_width=True,
            hide_index=True
        )
'''

with open(app_file, "w", encoding="utf-8") as f:
    f.write(app_code)

print("app.py updated successfully.")
print("Prediction button and output logic added.")

app.py updated successfully.
Prediction button and output logic added.


In [ ]:
process.terminate()

import time
time.sleep(2)

process = subprocess.Popen(
    ["streamlit", "run", app_file, "--server.port", "8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(5)

response = requests.get("http://localhost:8501")

print("Streamlit restarted successfully.")
print("Status code:", response.status_code)

Streamlit restarted successfully.
Status code: 200


In [ ]:
import importlib.util

spec = importlib.util.spec_from_file_location(
    "consumer_app",
    app_file
)

consumer_app = importlib.util.module_from_spec(spec)

spec.loader.exec_module(consumer_app)

predict_complaint_response = consumer_app.predict_complaint_response

print("Prediction function loaded successfully.")

2026-09-16 13:03:28.640 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-16 13:03:32.093 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-16 13:03:32.306 
  command:

    streamlit run /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-09-16 13:03:32.306 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-16 13:03:32.309 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-16 13:03:32.310 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-16 13:03:32.311 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

Prediction function loaded successfully.


In [ ]:
test_input = {
    "product": "Credit reporting or other personal consumer reports",
    "sub_product": "Credit reporting",
    "issue": "Problem with a company's investigation into an inaccurate report",
    "sub_issue": "Investigation took more than 30 days",
    "company": "Experian Information Solutions Inc.",
    "state": "GA",
    "submitted_via": "Web",
    "date_received": "2026-03-13",
    "consumer_complaint_narrative": ""
}

prediction, probability_table = predict_complaint_response(test_input)

print("Predicted response:")
print(prediction)

print("\nPrediction probabilities:")
display(probability_table)

Predicted response:
Closed with explanation

Prediction probabilities:


,Response Class,Probability
0,Closed with explanation,96.77
1,Closed with monetary relief,1.83
2,Closed with non-monetary relief,1.25
3,Untimely response,0.15


In [ ]:
for feature, categories in zip(
    encoder.feature_names_in_,
    encoder.categories_
):

_IncompleteInputError: incomplete input (4094624649.py, line 4)